<!-- nav -->
[← 2 · Answers you can trust](02_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [4 · Values with widths →](04_Values_With_Widths.ipynb)

# 3 · Constraints over collections

**The problem.** The latest reading isn't the whole story. A logger records the temperature every few minutes, and a
shipment that spent an hour at 12 °C may be back at 5 °C by the time it's checked. The real constraints are about *all*
the readings: every one in range, or at most two brief excursions, never above 12 °C. And about the loggers a shipment
carries: each one's battery must last the trip.

This case study writes constraints over lists and over an object's related objects, with **quantifiers**.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import { cold_chain, show } from "./toolkit.ts";

const store = cold_chain();
const steady = store.Shipment().id("S1").readings([4.0, 5.5, 6.1, 5.0]).create();
const blip = store.Shipment().id("S2").readings([4.0, 8.6, 9.1, 5.0]).create();
const cooked = store.Shipment().id("S3").readings([4.0, 12.5, 14.0, 13.2]).create();
const silent = store.Shipment().id("S4").create(); // no readings uploaded
const shipments = [steady, blip, cooked, silent];
const [self, c] = [E.variable("this"), E.variable("c")];
const table = (constraint: any) => Object.fromEntries(shipments.map((s: any) => [s.id, Evaluators.predicate(constraint, s)]));

## Step 1: a list is a collection

`readings` is a list property. Reading it gives a **collection**, which a handful of operations take: `count`, `item`
(by position, from 0), `in_`, `sum`, `min`, `max` and `unique`.

In [2]:
for (const expression of [self.readings, self.readings.count(), self.readings.item(0n), self.readings.max(),
  E.literal(9.1).in_(self.readings), self.readings.unique()]) {
  console.log(show(expression).padEnd(40), String(Evaluators.OfAny(expression, { this: blip })));
}

get(this, 'readings')                    [4.0, 8.6, 9.1, 5.0]


count(get(this, 'readings'))             4


item(get(this, 'readings'), 0)           4


max(get(this, 'readings'))               9.1


in(9.1, get(this, 'readings'))           true


unique(get(this, 'readings'))            true


Collections treat unknowns as everything else does. There's no item at position 9, so it's unknown,
not an error; the silent shipment has no readings at all, so everything about them is unknown:

In [3]:
for (const expression of [self.readings.item(9n), self.readings.count(), self.readings.max()]) {
  console.log(show(expression).padEnd(40), String(Evaluators.OfAny(expression, { this: blip })), String(Evaluators.OfAny(expression, { this: silent })));
}

item(get(this, 'readings'), 9)           null null


count(get(this, 'readings'))             4 null


max(get(this, 'readings'))               9.1 null


## Step 2: quantifiers

A **quantifier** binds a name to each item in turn, here `c`, and tests a body. `all` holds when the body holds for
every item, `any` when it holds for one, and `count_where` counts the items it holds for. (Python also reads them
from `all(...)`, `any(...)` and `sum(1 for ... if ...)` in a lambda.)

In [4]:
const every_in_range = self.readings.all("c", E.literal(2.0).le(c).and_(c.le(8.0)));
console.log(show(every_in_range));
console.log(table(every_in_range));

all(get(this, 'readings'), c => and(le(2.0, c), le(c, 8.0)))


{ S1: true, S2: false, S3: false, S4: null }


The steady shipment passes and the others fail; the silent one is unknown. The quantifier is a term of its own
kind, which binds `c` the way a let does, so `c` means nothing outside the body.

The real constraint allows brief excursions: at most two readings above 8 °C, and none above 12 °C. That's a count and a
maximum:

In [5]:
const excursions = self.readings.count_where("c", c.gt(8.0));
const allowed = excursions.le(2n).and_(self.readings.max().le(12.0));
console.log(show(allowed));
console.log(table(allowed));

and(le(count(get(this, 'readings'), c => gt(c, 8.0)), 2), le(max(get(this, 'readings')), 12.0))


{ S1: true, S2: true, S3: false, S4: null }


The blip is allowed now; the cooked shipment still isn't.

## Step 3: empty collections

An empty collection is data too, and quantifiers answer the way logic does: every item of nothing satisfies anything
(`all` is true), none satisfies something (`any` is false), and the count is 0. What has no answer is the greatest of
nothing, so `max` is unknown, and a constraint that relies on it says so:

In [6]:
const empty = store.Shipment().id("S5").readings([]).create();
for (const [name, constraint] of [["every_in_range", every_in_range], ["any above 8", self.readings.any("c", c.gt(8.0))],
  ["excursions", excursions], ["max", self.readings.max()], ["allowed", allowed]] as const) {
  console.log(name.padEnd(16), String(Evaluators.OfAny(constraint, { this: empty })));
}

every_in_range   true


any above 8      false


excursions       0


max              null


allowed          null


`allowed` for an empty list is unknown, not true: `excursions` is 0, which is fine, but the maximum is unknown. A
shipment that logged nothing hasn't shown it stayed cold, and the constraint doesn't pretend otherwise. If an empty log
*should* pass, say so explicitly, with `count`:

In [7]:
const lenient = self.readings.count().eq(0n).or_(allowed);
console.log(Evaluators.predicate(lenient, empty), Evaluators.predicate(lenient, cooked));

true false


## Step 4: an object's related objects

A shipment carries loggers, through the relation `Carries`, whose entries record the slot each logger rides in.
`entries(name)` gives an object's entries in a relation as a collection of **records**, which `get` reads like
objects: the linked logger, and the entry's own properties.

In [8]:
const trip = store.Shipment().id("S6").readings([5.0])
  .loggers((e: any) => e.slot(1n).logger((l: any) => l.serial("L-17").battery(80n)))
  .loggers((e: any) => e.slot(2n).logger((l: any) => l.serial("L-22").battery(15n))).create();
const e = E.variable("e");
for (const expression of [self.entries("loggers").count(), self.entries("loggers").item(1n).get("slot"),
  self.entries("loggers").item(1n).get("logger").get("serial")]) {
  console.log(show(expression).padEnd(70), String(Evaluators.OfAny(expression, { this: trip })));
}

count(entries(this, 'loggers'))                                        2


get(item(entries(this, 'loggers'), 1), 'slot')                         2


get(get(item(entries(this, 'loggers'), 1), 'logger'), 'serial')        L-22


`.get("logger")` is the writer of `get(e, 'logger')`; `e.logger` would be the same. So "every logger's battery
lasts the trip" is a quantifier over the entries:

In [9]:
const charged = self.entries("loggers").all("e", e.logger.battery.ge(20n));
console.log(show(charged));
console.log(Evaluators.predicate(charged, trip));
console.log(String(Evaluators.OfAny(self.entries("loggers").count_where("e", e.logger.battery.lt(20n)), { this: trip })));

all(entries(this, 'loggers'), e => ge(get(get(e, 'logger'), 'battery'), 20))


false


1


## Why it works this way

- **Quantifiers instead of loops.** A loop can do anything, including never finish. A quantifier visits each item of
  data once, so a constraint with quantifiers is still bounded, total and pure, and still a constraint a solver or a
  hardware checker could take.
- **Collections come from data.** There are no list literals: a collection is a list property, or an object's entries,
  or a variable the caller binds. That keeps constraints about the data, not little programs of their own.
- **The empty case is honest.** `all` of nothing is true because nothing violates it; the maximum of nothing doesn't
  exist. Constraints that need an answer for the empty case say what it is.

<!-- nav -->
[← 2 · Answers you can trust](02_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [4 · Values with widths →](04_Values_With_Widths.ipynb)